# Instrument Manager Test

Fill in your Yokogawa GPIB/VISA addresses and SGS100A TCPIP/VISA address, then run the cells from top to bottom.

This notebook starts with read-only checks: connect, `status`, `help`, and `limits`. The cells that change output values or turn outputs on are commented out on purpose.

In [ ]:
from QickworkspaceV2.instruments import BaseInstrumentManager

inst = BaseInstrumentManager()
inst.status

## Addresses

Edit these address strings before running the add cells. Leave `Q2_FLUX_ADDR = ""` if you only want to test one Yoko.

- Yoko: `GPIB0::1::INSTR`, `GPIB0::2::INSTR`, `USB0::...::INSTR`
- SGS100A: `TCPIP::192.168.0.10::INSTR`

In [ ]:
Q1_FLUX_ADDR = "GPIB0::1::INSTR"          # TODO: fill Q1 flux Yoko GPIB/VISA address
Q2_FLUX_ADDR = ""                         # Optional: fill Q2 flux Yoko address, or leave empty
PUMP_ADDR = "TCPIP::192.168.0.10::INSTR"  # TODO: fill SGS100A TCPIP/VISA address

# Lab safety limits. Adjust to your wiring/device before setting values.
Q1_FLUX_LIMITS = {
    "current": (-3e-3, 3e-3),
    "voltage": (-1.0, 1.0),
}

Q2_FLUX_LIMITS = {
    "current": (-3e-3, 3e-3),
    "voltage": (-1.0, 1.0),
}

PUMP_LIMITS = {
    "frequency": (1e6, 20e9),
    "power": (-80, 10),
}

## Add Instruments

These cells connect and register the instruments. They should not change the output level by themselves.

The names (`q1_flux`, `q2_flux`, `pump`) are how you choose which physical instrument to control later.

In [ ]:
q1_flux = inst.add_yoko(
    "q1_flux",
    Q1_FLUX_ADDR,
    limits=Q1_FLUX_LIMITS,
    current_ramp_step=1e-8,
    voltage_ramp_step=1e-5,
    ramp_interval=0.01,
)

q2_flux = None
if Q2_FLUX_ADDR:
    q2_flux = inst.add_yoko(
        "q2_flux",
        Q2_FLUX_ADDR,
        limits=Q2_FLUX_LIMITS,
        current_ramp_step=1e-8,
        voltage_ramp_step=1e-5,
        ramp_interval=0.01,
    )

inst.status

In [ ]:
pump = inst.add_sgs100(
    "pump",
    PUMP_ADDR,
    limits=PUMP_LIMITS,
)

inst.status

## Status, Help, Limits

In [ ]:
inst.status

In [ ]:
inst.help()

In [ ]:
inst.limits()

## Read Current Values

These are read/query checks only.

In [ ]:
print("Q1 flux IDN:", q1_flux.idn())
print("Q1 flux value:", inst.value("q1_flux"))
print("Q1 flux ramp:", inst.ramp("q1_flux"))

if q2_flux is not None:
    print("Q2 flux IDN:", q2_flux.idn())
    print("Q2 flux value:", inst.value("q2_flux"))
    print("Q2 flux ramp:", inst.ramp("q2_flux"))

print("Pump IDN:", pump.idn())
print("Pump value:", inst.value("pump"))
print("Pump snapshot:", pump.snapshot())

## Optional Write Tests

Only run these after confirming the ranges above are safe for your setup.

In [ ]:
# Yoko value/ramp tests. Uncomment only when ready.
# inst.configure_ramp("q1_flux", current_step=1e-8, voltage_step=1e-5, interval=0.01)
# inst.set_value("q1_flux", 0.0, mode="current")
# inst.value("q1_flux")

# If you registered q2_flux:
# inst.configure_ramp("q2_flux", current_step=1e-8, voltage_step=1e-5, interval=0.01)
# inst.set_value("q2_flux", 0.0, mode="current")
# inst.value("q2_flux")

# inst.status

In [ ]:
# SGS value/frequency test. Uncomment only when ready.
# inst.set("pump", "frequency", 6.0e9)
# inst.set_value("pump", -40)
# inst.off("pump")
# inst.status

## Close Connections

In [ ]:
# Run this when finished.
# inst.close()